# 04 First canine-only baseline models

Purpose: run simple canine-only baseline models after endpoints are confirmed.

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

print("Project root:", PROJECT_ROOT)
print("Raw data dir:", RAW_DIR)
print("Processed data dir:", PROCESSED_DIR)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, RocCurveDisplay, PrecisionRecallDisplay

analysis_path = PROCESSED_DIR / "GSE238110_DOG2_analysis_top5000var.csv"
analysis = pd.read_csv(analysis_path, index_col=0)

print("Analysis shape:", analysis.shape)
display(analysis.head())

In [ ]:
# Edit this after clinical endpoint review.
TARGET_COL = "metastasis_event"

if TARGET_COL not in analysis.columns:
    raise ValueError(f"{TARGET_COL} was not found. Set TARGET_COL to an available binary endpoint.")

y = analysis[TARGET_COL]
valid = y.notna()
y = y.loc[valid].astype(int)

known_endpoint_cols = ["metastasis_event", "os_time", "os_event", "dfi_time", "dfi_event"]
gene_cols = [c for c in analysis.columns if c not in known_endpoint_cols]
X = analysis.loc[valid, gene_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

print("X:", X.shape)
print("Class counts:")
display(y.value_counts())

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegressionCV(
        Cs=10,
        cv=5,
        penalty="elasticnet",
        solver="saga",
        l1_ratios=[0.1, 0.5, 0.9],
        scoring="roc_auc",
        max_iter=5000,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pred = cross_val_predict(model, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print("ROC-AUC:", roc_auc_score(y, pred))
print("PR-AUC:", average_precision_score(y, pred))
print("Balanced accuracy at 0.5:", balanced_accuracy_score(y, pred >= 0.5))

RocCurveDisplay.from_predictions(y, pred)
plt.title("Canine-only Elastic Net logistic ROC")
plt.show()

PrecisionRecallDisplay.from_predictions(y, pred)
plt.title("Canine-only Elastic Net logistic PR curve")
plt.show()